# EHAM A-CDM Decision Hub - Demo Notebook

Five-act walkthrough of the A-CDM Decision Hub on top of `ACDM_DEMO.EHAM` (320 flights, day of 2026-10-14). Each act uses a different RAI reasoner family on the SAME PyRel ontology.

| Act | Reasoner    | Question |
|-----|-------------|----------|
| 1   | Rules        | TOBT compliance audit (MS12 +/-5 min) |
| 2   | Graph        | KL1234 rotation cascade |
| 3   | Heuristic    | MS5 gate-conflict ranking |
| 4   | Prescriptive | TSAT re-sequence under storm |
| 5   | Persistent rule | Add 'preserve KL high-pax' rule and re-solve |

Run from project root:  `.venv/bin/python -m jupyter lab rai_code/manual/eham_acdm_demo.ipynb`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

# Disarm PyRel's running-loop guard so synchronous Problem.solve() works
# inside Jupyter (mirrors supply_chain_demo.demo_helpers.setup_jupyter_compat).
import relationalai.client as _ra_client
import relationalai.services.reasoners.client as _ra_reasoners_client
_noop = lambda *a, **k: None
_ra_client.raise_if_running_event_loop = _noop
_ra_reasoners_client.raise_if_running_event_loop = _noop

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

# Importing the ontology triggers all model.define() statements.
from rai_code.manual.eham_acdm import (
    Flight, Stand, Operator, GroundHandler, model,
    feeds_callsign, slot_blocks, shares_stand,
    Departure, Arrival, TOBTViolation, StormWindowDeparture, PreservedFlight,
)
from rai_code.manual.demo_queries import (
    q1_tobt_violations_by_handler,
    q2_rotation_cascade_from,
    q3_ms5_conflict_ranking,
    q4_tsat_resequence_under_storm,
    q5_tsat_resequence_with_preservation,
)
print("imports ok")


## Act 1 - Rules: TOBT compliance audit

> **The question (operator types):** 'Show me every flight in the last four hours where actual ready time deviated from TOBT by more than five minutes, broken down by ground handler.'

The `TOBTViolation` derived concept encodes ICAO MS12 once at the ontology layer. The query just counts.

In [ ]:
df1 = q1_tobt_violations_by_handler()
df1

In [ ]:
fig = px.bar(
    df1, x='handler', y='violations',
    color='avg_deviation_min',
    color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
    title='TOBT violations by handler (4h window before 14:30; |ARDT - TOBT| > 5 min)',
    labels={'avg_deviation_min': 'avg deviation (min)'},
    text='violations',
)
fig.update_traces(textposition='outside')
fig.update_layout(height=380, xaxis_title='handler', yaxis_title='violation count')
fig.show()

**What the chart shows.** KLG and AGS are running positive ARDT-vs-TOBT deviations (flights calling ready *late*). DNATA is negative - calling ready *early*, which inflates demand on the TSAT calculator. Both are A-CDM hygiene issues, surfaced from a single derived concept defined once at the ontology layer.

## Act 2 - Graph: KL1234 rotation cascade

> **The question:** 'KL1234 inbound from KJFK is 35 minutes late at final approach. Trace the rotation impact across the next 6 hours.'

The cascade graph is the union of three edge types in the same ontology:
- `feeds_callsign` - explicit aircraft rotation (KL1234 -> KL1235, KL1402 -> KL1601)
- `slot_blocks` - operational pushback contention (KL1235 -> HV5821, KL1402 -> AF1241)
- `shares_stand` - stand-occupancy overlap (E18 carries the KL chain)

The Graph reasoner runs reachability over the union.

In [ ]:
df2 = q2_rotation_cascade_from('KL1234')
df2

In [ ]:
# Visualize the cascade as a directed graph. Pull all (src, dst) edges, lay
# them out with networkx, then render with plotly.
from relationalai.semantics.std import aggregates as aggs
src = Flight.ref(); dst = Flight.ref()
rot = (
    model.where(feeds_callsign(src, dst))
    .select(src.callsign.alias('src'), dst.callsign.alias('dst'))
    .to_df()
)
rot['kind'] = 'rotation'
src = Flight.ref(); dst = Flight.ref()
sb = (
    model.where(slot_blocks(src, dst))
    .select(src.callsign.alias('src'), dst.callsign.alias('dst'))
    .to_df()
)
sb['kind'] = 'slot_block'
src = Flight.ref(); dst = Flight.ref()
ss = (
    model.where(shares_stand(src, dst), src.sobt < dst.sobt)
    .select(src.callsign.alias('src'), dst.callsign.alias('dst'))
    .to_df()
)
ss['kind'] = 'stand'

cascade_callsigns = set(df2['to'])
all_edges = pd.concat([rot, sb, ss], ignore_index=True)
cascade_edges = all_edges[
    all_edges['src'].isin(cascade_callsigns)
    & all_edges['dst'].isin(cascade_callsigns)
].drop_duplicates(subset=['src', 'dst', 'kind'])

G = nx.DiGraph()
for _, r in cascade_edges.iterrows():
    G.add_edge(r['src'], r['dst'], kind=r['kind'])
for c in cascade_callsigns:
    G.add_node(c)
pos = nx.spring_layout(G, seed=42, k=1.4)

edge_x, edge_y, edge_color = [], [], []
kind_palette = {'rotation': '#1f77b4', 'slot_block': '#d62728', 'stand': '#9467bd'}
edge_traces = []
for kind, color in kind_palette.items():
    xs, ys = [], []
    for u, v, data in G.edges(data=True):
        if data.get('kind') != kind:
            continue
        x0, y0 = pos[u]; x1, y1 = pos[v]
        xs += [x0, x1, None]; ys += [y0, y1, None]
    if xs:
        edge_traces.append(go.Scatter(
            x=xs, y=ys, mode='lines',
            line=dict(width=2, color=color),
            hoverinfo='none', name=kind,
        ))
node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_text = list(G.nodes())
is_seed = ['KL1234' == n for n in G.nodes()]
node_color = ['#ff7f0e' if s else '#cccccc' for s in is_seed]
node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    marker=dict(size=28, color=node_color, line=dict(width=2, color='#333')),
    text=node_text, textposition='middle center',
    name='flight', textfont=dict(size=10),
)
fig = go.Figure(data=edge_traces + [node_trace])
fig.update_layout(
    title='KL1234 rotation cascade (rotation = blue, slot_block = red, stand = purple)',
    showlegend=True, height=560, margin=dict(l=10, r=10, t=50, b=10),
    xaxis=dict(visible=False), yaxis=dict(visible=False),
)
fig.show()

**What the chart shows.** From a single late ALDT input (KL1234), the agent reaches six downstream flights across three carriers (KL, HV, AF), two terminals, and one ATFM slot at risk. The Eurocontrol Network Manager wants to know this an hour before the current systems surface it.

## Act 3 - Heuristic: MS5 gate-conflict ranking

> **The question:** 'Looking at all inbound flights currently between MS5 final approach and MS6 landing - rank them by probability of arrival gate conflict in the next 30 minutes.'

The talk-track Predictive reasoner is in early access; per its disclaimer we use a deterministic fit-score, expressed entirely as derived properties in PyRel:

```
ms5_score = 0.40 * (30 - minutes_to_landing) / 30
          + 0.35 * pax_connections / 150
          + ms5_pier_bonus   (0.15 if pier in {M, G, H} else 0)
          - ms5_wtc_penalty  (0.20 if wtc not in {H, M} else 0)
```

In [ ]:
df3 = q3_ms5_conflict_ranking()
df3

In [ ]:
df3_sorted = df3.copy()
df3_sorted['highlight'] = df3_sorted['inbound'].isin(['DL0036', 'KL0691', 'AF1641', 'BA0432', 'KL1234'])
fig = px.bar(
    df3_sorted.head(11), x='p_conflict', y='inbound', orientation='h',
    color='pier', text='p_conflict',
    hover_data=['current_stand', 'pax_conn', 'wtc', 'minutes_to_land'],
    title='MS5 gate-conflict ranking (deterministic heuristic, all weights in PyRel)',
)
fig.update_traces(texttemplate='%{x:.2f}', textposition='outside')
fig.update_layout(
    height=480, yaxis=dict(categoryorder='total ascending', title=None),
    xaxis=dict(title='ms5_score'),
)
fig.show()

**What the chart shows.** Eleven arrival candidates ranked by their MS5 fit. The five talk-track flights (KL1234, DL0036, KL0691, AF1641, BA0432) all surface. The output is a *ranked queue* (not a yes/no flag) so the stand planner doesn't swap stands on every false positive.

## Act 4 - Prescriptive: TSAT re-sequence under storm

> **The scenario:** 'Forecast says a thunderstorm cell crosses 18C/27 from 15:00 to 17:00. We have 47 outbound flights with TSAT in that window. Solve the optimal sequence.'

Decision variables: binary `FlightSlot.assign_base` over 47 flights x 120 1-minute slots = ~5,600 binaries. HiGHS solves in under 5 seconds.

Constraints encoded in the Problem:
1. Each flight assigned to exactly one slot.
2. Each slot holds at most one flight (single-runway).
3. No flight pushed before its SOBT.
4. Per-pier pushback contention <= 2 simultaneous.

Objective: minimize total delay weighted by `(pax_connections + 20 * atfm_penalty)`.

In [ ]:
df4, si4 = q4_tsat_resequence_under_storm()
print(f'status: {si4.termination_status}')
print(f'objective: {si4.objective_value:,.0f} (weighted minute-pax)')
print(f'solve time: {si4.solve_time_sec:.2f}s')
print(f'flights placed: {len(df4)}')
df4.head(20)

In [ ]:
# Gantt: each flight as a bar on the storm window timeline.
df4_gantt = df4.copy()
# RAI returns integer columns as Int128Array; cast to plain int64 so pandas
# arithmetic and pd.to_timedelta work without tripping the Int128 __add__.
for col in ('minute_offset', 'abs_min', 'sobt_min', 'delay_min'):
    if col in df4_gantt.columns:
        df4_gantt[col] = df4_gantt[col].astype('int64')
df4_gantt['start_min'] = df4_gantt['minute_offset']
df4_gantt['end_min'] = df4_gantt['minute_offset'] + 1
df4_gantt['start'] = pd.Timestamp('2026-10-14 15:00') + pd.to_timedelta(df4_gantt['start_min'], unit='m')
df4_gantt['end'] = pd.Timestamp('2026-10-14 15:00') + pd.to_timedelta(df4_gantt['end_min'], unit='m')
fig = px.timeline(
    df4_gantt.sort_values('minute_offset'),
    x_start='start', x_end='end', y='callsign',
    color='delay_min',
    color_continuous_scale='RdYlGn_r',
    hover_data=['op', 'pier', 'pax_conn', 'delay_min'],
    title='Act 4: optimized 18L departure sequence (storm window 15:00 - 17:00)',
)
fig.update_yaxes(categoryorder='total descending')
fig.update_layout(height=900, xaxis_title='time', yaxis_title='callsign')
fig.show()


In [ ]:
# Delay distribution.
fig = px.histogram(df4, x='delay_min', nbins=30, title='Act 4 delay distribution (min vs SOBT)')
fig.update_layout(height=320)
fig.show()
total_delay = df4['delay_min'].sum()
print(f'total raw delay: {total_delay} min')
print(f'avg delay/flight: {df4["delay_min"].mean():.1f} min')
print(f'max delay: {df4["delay_min"].max()} min')
print(f'flights with delay = 0: {(df4["delay_min"] == 0).sum()} / {len(df4)}')

## Act 5 - Persistent rule: operator adds a preservation rule

> **The operator types:** 'Never delay KL flights with more than 80 connecting pax by more than 8 minutes from SOBT for runway re-sequencing.'

Two things to land:
1. The rule becomes a `PreservedFlight(Flight)` derived concept and an aggregate-delay constraint that lives in the SAME ontology.
2. The rule persists across solves - it isn't a comment in someone's runbook.

In [ ]:
df5, si5 = q5_tsat_resequence_with_preservation()
print(f'status: {si5.termination_status}')
print(f'objective: {si5.objective_value:,.0f}')
print(f'solve time: {si5.solve_time_sec:.2f}s')
df5.head(20)

In [ ]:
# Side-by-side: Q4 vs Q5 delays.
join = df4[['callsign', 'pax_conn', 'delay_min']].rename(columns={'delay_min': 'act4'}).merge(
    df5[['callsign', 'delay_min']].rename(columns={'delay_min': 'act5'}),
    on='callsign', how='outer',
)
join['delta'] = join['act5'] - join['act4']
join = join.sort_values('delta')
join.head(20)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Act 4 delay (no preservation)', 'Act 5 delay (with preservation)'))
for col, name, df in ((1, 'act4', df4), (2, 'act5', df5)):
    fig.add_trace(
        go.Bar(
            x=df['callsign'], y=df['delay_min'],
            marker_color=['#ff7f0e' if c == 'KL691' else '#1f77b4' for c in df['callsign']],
            name=name,
        ),
        row=1, col=col,
    )
fig.update_layout(height=420, showlegend=False, title_text='Per-flight delay; KL691 highlighted')
fig.update_xaxes(tickangle=-60)
fig.show()

kl691 = join[join['callsign'] == 'KL691']
if not kl691.empty:
    print(f"KL691 -> RJTT (137 connecting pax): Act 4 = {int(kl691['act4'].iloc[0])} min, Act 5 = {int(kl691['act5'].iloc[0])} min")

**Closing.** Four questions, four reasoners, one semantic model, one ICAO procedure encoded as logic. Adding a new constraint took ~10 lines of PyRel and didn't break anything. The next operator who joins inherits the rule - no tribal knowledge, no Excel sheet, no PDF.